In [ ]:
# https://www.kaggle.com/code/ybabakhin/mistral-7b-prompt-predict-private/edit
!pip install -U accelerate --no-index --find-links ../input/llm-detect-pip/
!pip install -U bitsandbytes --no-index --find-links ../input/llm-detect-pip/
# !pip install -U transformers --no-index --find-links ../input/llm-detect-pip/

In [ ]:
!pip install /kaggle/input/prompt-pip/transformers-4.38.2-py3-none-any.whl

In [ ]:
!pip install -Uq /kaggle/input/sentence-transformers-2-4-0/sentence_transformers-2.4.0-py3-none-any.whl

In [ ]:
from datasets import load_dataset
from tqdm.notebook import tqdm
import torch
import pandas as pd
from glob import glob
import numpy as np
import os

from transformers import AutoModelForCausalLM, AutoTokenizer
import itertools
import random
import argparse
import os
import pandas as pd
import numpy as np
from string import Template
from pathlib import Path
import time
import torch
from tqdm.auto import tqdm
import gc

os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from pathlib import Path

data_path = Path('/kaggle/input/llm-prompt-recovery')

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    test = pd.read_csv(data_path / 'test.csv', index_col='id')
    test["rewrite_prompt"] = "-"
else:
    test = pd.read_csv("/kaggle/input/prompt-val/tmp_val.csv")
    
test = test.fillna("")
test.head()

In [ ]:
import re
patterns = [
    r"^.?[hH]ere is.*\n\n",
    r"^.?[hH]ere \'s.*\n\n",
    r"^.?[Ss]ure, here.*\n\n",
]
t = []
for filtered in test["rewritten_text"].values:
    for pattern in patterns:
        filtered = re.sub(pattern, "", filtered)
    t.append(filtered)
    
test["rewritten_text_v2"] = t

In [ ]:
test.to_parquet("test.pq", index=False)

In [ ]:
import os

os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
%%writefile public_sol.py


import torch
import random
import numpy as np
import pandas as pd
import gc
import time
import argparse
import os
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


if __name__ == "__main__":

    ap = argparse.ArgumentParser()
    ap.add_argument("--device", type=str, required=True)
    args = ap.parse_args()
    
    DEVICE = args.device
    
    #https://github.com/Lightning-AI/lit-gpt/issues/327
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_flash_sdp(False)

    if (not torch.cuda.is_available()): print("Sorry - GPU required!")

    import logging
    logging.getLogger('transformers').setLevel(logging.ERROR)

    #this can help speed up inference
    max_new_tokens = 30

    #output test is trimmed according to this
    max_sentences_in_response = 1

    model_name = '/kaggle/input/mistral-7b-it-v02'
    tokenizer = AutoTokenizer.from_pretrained(model_name) 

    # Load base model(Mistral 7B)
    bnb_config = BitsAndBytesConfig(  
        load_in_4bit= True,
        bnb_4bit_quant_type= "nf4",
        bnb_4bit_compute_dtype= torch.float16,
        bnb_4bit_use_double_quant= False,
    )

    model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            torch_dtype=torch.float16,
            device_map=DEVICE,
            trust_remote_code=True,
    )

    #original text prefix
    orig_prefix = "Original Text:"

    #mistral "response"
    llm_response_for_rewrite = "Provide the new text and I will tell you what new element was added or change in tone was made to improve it. I will avoid mentioning names of characters. It is crucial no person, place or thing from the original text be mentioned, but only the overall type of the text. For example - I will say 'Improve this report by turning it into a philosophical essay.' - If not sure about the original type I will just say 'Improve this text [...]'.  If the original text mentions a specific idea, person, place, or thing - I will not mention it in my answer.  For example if there is a 'dog' or 'office' in the original text - the word 'dog' or 'office' must not be in my response. My answer will be a single sentence."

    #modified text prefix
    rewrite_prefix = "Re-written Text:"

    #provided as start of Mistral response (anything after this is used as the prompt)
    #providing this as the start of the response helps keep things relevant
    response_start = "The request was: "

    #added after response_start to prime mistral
    #"Improve this" or "Improve this text" resulted in non-answers.  
    #"Improve this text by" seems to product good results
    response_prefix = "Improve this"

    #these will all be given to Mistral before each and every prompt
    #original_text
    #rewritten_text
    #prompt

    examples_sequences = [
        (
            "Andrew and Jane, a perfect pair,\nOn their date, love filled the air.",
            "Andrew and Jane, a perfect pair, their bond tainted by a shared addiction. The intoxicating aroma of their favorite drug permeated the air, a bittersweet scent of destruction and desperation.",
            "Improve this text by focusing on addiction."
        ),
        (
            "In the small town of Oakwood, Lester's execution was scheduled for tomorrow morning. The community held its breath, torn between justice and mercy. As the sun set, whispers of hope and despair filled the air, leaving a heavy weight on everyone's hearts.",
            "In the small town of Oakwood, the community was preparing for their annual summer festival tomorrow. The air was filled with excitement and anticipation, as people prepared to come together and enjoy a day of fun and festivities. As the sun set, the town erupted in a symphony of laughter and music, creating a joyful atmosphere for all.",
            "Improve this text by making it joyful."
        ),
        (
            "Hi Ethel, just wanted to let you know that the new chair for the conference room has arrived.",
            "Hi Ethel, prepare for the arrival of the grandest throne in the conference room, a seat of power and wisdom, fit for a queen.",
            "Improve this text by exaggerating the importance of the chair."
        ),
        (
            "Rachel's binoculars,\nZoom in on nature's show,\nBirds soaring, leaves rustling,\nA world unseen, now aglow",
            "Rachel\'s binoculars cast a gaze upon the sky,\nZooming into nature\'s dance,\nBirds soar through clouds, leaves dance,\nA hidden world unfolds, bathed in light.",
            "Improve this text by making it poetic."
        ),
#         (
#             "Memo\n\nTo: All Staff\nFrom: [Your Name]\nDate: [Today's Date]\n\nSubject: Metal Recycling Program\n\nI am pleased to announce the launch of our new metal recycling program at Wilbur Manufacturing. Starting next week, we will have designated bins placed throughout the facility for employees to dispose of any metal waste, such as aluminum cans, steel scraps, and copper wires. This initiative aligns with our commitment to sustainability and reducing our environmental footprint. Let's all contribute to a greener future by participating in this program. Thank you for your cooperation.\n\nBest regards,\n[Your Name]",
#             "Memo\n\nTo: All Staff\nFrom: [Your Name]\nDate: [Today's Date]\n\nSubject: Metal Recycling Program\n\nI am pleased to announce the launch of our new metal recycling program at Wilbur Manufacturing, um... Starting next week, we will have designated bins placed throughout the facility for employees to dispose of any metal waste, such as aluminum cans, steel scraps, and copper wires. This initiative aligns with our commitment to sustainability and reducing our environmental footprint, um... Let's all contribute to a greener future by participating in this program. Thank you for your cooperation.\n\nBest regards,\n[Your Name]",
#             'Improve this text by frequently adding the phrase "um".'
#         ),
        (
            "Hey Lois! Just wanted to let you know that grandma's birthday is coming up next week. I was thinking we could all get together for a small celebration at her place. Let me know if you're available and we can start planning. Can't wait to see grandma's smile!",
            "Hey Lo, what's up! Grandma's b-day is coming up next week, so I was thinking we could throw her a small party at her crib. Let me know if you're down to help out and we can start planning. I'm super excited to see grandma's smile!",
            "Improve this text by translating into contemporary slang."
        ),
        (
            "Hey there! Just a heads up: our friendly dog may bark a bit, but don't worry, he's all bark and no bite!",
            "Warning: Protective dog on premises. May exhibit aggressive behavior. Ensure personal safety by maintaining distance and avoiding direct contact.",
            "Improve this memo by converting it into a warning."
        ),

        (
            "A lunar eclipse happens when Earth casts its shadow on the moon during a full moon. The moon appears reddish because Earth's atmosphere scatters sunlight, some of which refracts onto the moon's surface. Total eclipses see the moon entirely in Earth's shadow; partial ones occur when only part of the moon is shadowed.",
            "Yo check it, when the Earth steps in, takes its place, casting shadows on the moon's face. It's a full moon night, the scene's set right, for a lunar eclipse, a celestial sight. The moon turns red, ain't no dread, it's just Earth's atmosphere playing with sunlight's thread, scattering colors, bending light, onto the moon's surface, making the night bright. Total eclipse, the moon's fully in the dark, covered by Earth's shadow, making its mark. But when it's partial, not all is shadowed, just a piece of the moon, slightly furrowed. So that's the rap, the lunar eclipse track, a dance of shadows, with no slack. Earth, moon, and sun, in a cosmic play, creating the spectacle we see today.",
            "Improve this text by altering the style into a rap."
        ),
#         (
#             "The park was empty, save for a solitary figure sitting on a bench, lost in thought. The quiet of the evening was punctuated only by the occasional rustle of leaves, offering a moment of peace in the chaos of city life.",
#             "Beneath the cloak of twilight, the park transformed into a realm of solitude and reflection. There, seated upon an ancient bench, was a lone soul, a guardian of secrets, enveloped in the serenity of nature's whispers. The dance of the leaves in the gentle breeze sang a lullaby to the tumult of the urban heart.",
#             "Improve this text by transforming it to be more poetic."
#         ),
        (
            "The startup team sat in the dimly lit room, surrounded by whiteboards filled with ideas, charts, and plans. They were on the brink of launching a new app designed to make home maintenance effortless for homeowners. The app would connect users with local service providers, using a sophisticated algorithm to match needs with skills and availability. As they debated the features and marketing strategies, the room felt charged with the energy of creation and the anticipation of what was to come.",
            "In the quiet before dawn, a small group of innovators gathered, their mission: to simplify home maintenance through technology. But their true journey began with the unexpected addition of Max, a talking car with a knack for solving problems. 'Let me guide you through this maze of decisions,' Max offered, his dashboard flickering to life.",
            "Improve this text by adding a talking car."
        ),
    ]

    def remove_numbered_list(text):
        final_text_paragraphs = [] 
        for line in text.split('\n'):
            # Split each line at the first occurrence of '. '
            parts = line.split('. ', 1)
            # If the line looks like a numbered list item, remove the numbering
            if len(parts) > 1 and parts[0].isdigit():
                final_text_paragraphs.append(parts[1])
            else:
                # If it doesn't look like a numbered list item, include the line as is
                final_text_paragraphs.append(line)

        return '  '.join(final_text_paragraphs)


    #trims LLM output to just the response
    def trim_to_response(text):
        terminate_string = "[/INST]"
        text = text.replace('</s>', '')
        #just in case it puts things in quotes
        text = text.replace('"', '')
        text = text.replace("'", '')
        text = text.replace("Improve this ", "")
        # text = text.replace("text by ", "")

        last_pos = text.rfind(terminate_string)
        return text[last_pos + len(terminate_string):] if last_pos != -1 else text

    #looks for response_start / returns only text that occurs after
    def extract_text_after_response_start(full_text):
        parts = full_text.rsplit(response_start, 1)  # Split from the right, ensuring only the last occurrence is considered
        if len(parts) > 1:
            return parts[1].strip()  # Return text after the last occurrence of response_start
        else:
            return full_text  # Return the original text if response_start is not found


    #trims text to requested number of sentences (or first LF or double-space sequence)
    def trim_to_first_x_sentences_or_lf(text, x):
        if x <= 0:
            return ""

        # Any double-spaces dealt with as linefeed
        text = text.replace("  ", "\n")

        # Split text at the first linefeed
        text_chunks = text.split('\n', 1)
        first_chunk = text_chunks[0]

        # Split the first chunk into sentences, considering the space after each period
        sentences = [sentence.strip() for sentence in first_chunk.split('.') if sentence]

        # If there's a linefeed, return the text up to the first linefeed
        if len(text_chunks) > 1:
            # Check if the first chunk has fewer sentences than x, and if so, just return it
            if len(sentences) < x:
                trimmed_text = first_chunk
            else:
                # Otherwise, trim to x sentences within the first chunk
                trimmed_text = '. '.join(sentences[:x]).strip()
        else:
            # If there's no linefeed, determine if the number of sentences is less than or equal to x
            if len(sentences) <= x:
                trimmed_text = '. '.join(sentences).strip()  # Ensure space is preserved after periods
            else:
                # Otherwise, return the first x sentences, again ensuring space after periods
                trimmed_text = '. '.join(sentences[:x]).strip()

        # Add back the final period if it was removed and the text needs to end with a sentence.
        if len(sentences) > 0 and not trimmed_text.endswith('.'):
            trimmed_text += '.'

        return trimmed_text

    def get_prompt(orig_text, transformed_text):

        messages = []

        # Append example sequences
        for example_text, example_rewrite, example_prompt in examples_sequences:
            messages.append({"role": "user", "content": f"{orig_prefix} {example_text}"})
            messages.append({"role": "assistant", "content": llm_response_for_rewrite})
            messages.append({"role": "user", "content": f"{rewrite_prefix} {example_rewrite}"})
            messages.append({"role": "assistant", "content": f"{response_start} {example_prompt}"})

        #actual prompt
        messages.append({"role": "user", "content": f"{orig_prefix} {orig_text}"})
        messages.append({"role": "assistant", "content": llm_response_for_rewrite})
        messages.append({"role": "user", "content": f"{rewrite_prefix} {transformed_text}"})
        messages.append({"role": "assistant", "content": f"{response_start} {response_prefix}"})

        #give it to Mistral
        model_inputs = tokenizer.apply_chat_template(messages, return_tensors="pt")
        # remove </s>
        model_inputs = model_inputs[:, :-1]
        model_inputs = model_inputs.to("cuda") 
        generated_ids = model.generate(
            model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

        #decode and trim to actual response
        decoded = tokenizer.batch_decode(generated_ids)
        just_response = trim_to_response(decoded[0])        
        final_text = extract_text_after_response_start(just_response)

        #mistral has been replying with numbered lists - clean them up....
        final_text = remove_numbered_list(final_text)

        #mistral v02 tends to respond with the input after providing the answer - this tries to trim that down
        final_text = trim_to_first_x_sentences_or_lf(final_text, max_sentences_in_response)

        return final_text

    from pathlib import Path

    data_path = Path('/kaggle/input/llm-prompt-recovery')

    if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
        test = pd.read_csv(data_path / 'test.csv', index_col='id')
        test["rewrite_prompt"] = "-"
    else:
        test = pd.read_csv("/kaggle/input/prompt-val/tmp_val.csv")

    test = test.fillna("")

    if DEVICE == "cuda:0":
        test = test.iloc[:len(test)//2].copy()
    else:
        test = test.iloc[len(test)//2:].copy()

    # End timing
    start_time = time.time()
    
    answers = []
    for idx, row in tqdm(test.iterrows(), total=len(test)):
        answer = get_prompt(row['original_text'], row['rewritten_text'])
        answers.append(answer)
        if idx <= 4:
            print(f"Actual Prompt: {row['rewrite_prompt']}")
            print(f"Predicted Prompt: {answer}")

    answers = pd.DataFrame(answers, columns=["rewrite_prompt_pred"])
    answers.to_csv(f"fs_test_answers_{DEVICE.split(':')[-1]}.csv", index=False)

    # Calculate and print the elapsed time
    end_time = time.time()

    elapsed_time_per_test = (end_time - start_time) / len(test)

    print(f"\n\n{elapsed_time_per_test} seconds per prediction.")
    print(f"Estimated {(elapsed_time_per_test * 750) / 3600} hours for 750 tests.")

In [ ]:
%%writefile run.sh

python public_sol.py --device "cuda:0" &
python public_sol.py --device "cuda:1" &

wait 
echo "All done"

In [ ]:
!sh run.sh

In [ ]:
answers0 = pd.read_csv("fs_test_answers_0.csv")
answers1 = pd.read_csv("fs_test_answers_1.csv")
answers_fs = pd.concat([answers0, answers1]).rewrite_prompt_pred.values

test["answers_fs"] = answers_fs

test.to_parquet("test.pq", index=False)

In [ ]:
answers_fs

In [ ]:
%%writefile run_embedding.py

from datasets import load_dataset
from tqdm.notebook import tqdm
import torch
import pandas as pd
from glob import glob
import numpy as np
import os

from transformers import AutoModelForCausalLM, AutoTokenizer
import itertools
import random
import argparse
import os
import pandas as pd
import numpy as np
from string import Template
from pathlib import Path
import time
import torch
from tqdm.auto import tqdm
import gc

os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if __name__ == "__main__":

    ap = argparse.ArgumentParser()
    ap.add_argument("--model", type=str, required=True)
    args = ap.parse_args()

    test = pd.read_parquet("test.pq")

    if args.model == "danube":
        model_name = "/kaggle/input/ybabakhin-gen-data-all"  # either local folder or huggingface model name

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map={"": "cuda:0"},
            trust_remote_code=True,
        )
        model.cuda().eval()

        head_weights = torch.load("/kaggle/input/ybabakhin-gen-data-all/classification_head.pth", map_location="cuda")
    elif args.model == "mistral":
        model_name = "/kaggle/input/ybabakhin-gen-data-all-data-v8-mistral"  # either local folder or huggingface model name

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
        )
        model.eval()

        head_weights = torch.load("/kaggle/input/ybabakhin-gen-data-all-data-v8-mistral/classification_head.pth", map_location="cuda")
    else:
        raise ValueError()
    
    tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            use_fast=True,
            trust_remote_code=True,
        )
    
    # settings can be arbitrary here as we overwrite with saved weights
    head = torch.nn.Linear(1, 1, bias=False).to("cuda")
    head.weight.data = head_weights
    print(head.weight.shape)

    def format_prompt(row):
        prompt = f"{row['original_text']}\n###\n{row['rewritten_text']}"
        return prompt

    def get_answer(row):

        prompt = format_prompt(row)
        inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")

        out = model(**inputs).logits

        logits = head(out[:,-1]).detach().cpu().numpy()
        del out
        return logits

    torch.backends.cuda.enable_mem_efficient_sdp(False)

    all_logits = []
    for _,row in tqdm(test.iterrows(), total=len(test)):
        logits = get_answer(row)
        all_logits.append(logits)

    from sklearn.preprocessing import normalize
    all_logits = np.concatenate(all_logits)
    all_logits = normalize(all_logits, norm="l2", axis=1)

    np.save(f"logits_embedding_{args.model}", all_logits)

In [ ]:
!python run_embedding.py --model=danube

In [ ]:
!python run_embedding.py --model=mistral

In [ ]:
%%writefile run_llm1.py

from datasets import load_dataset
from tqdm.notebook import tqdm
import torch
import pandas as pd
from glob import glob
import numpy as np
import os

from transformers import AutoModelForCausalLM, AutoTokenizer
import itertools
import random
import argparse
import os
import pandas as pd
import numpy as np
from string import Template
from pathlib import Path
import time
import torch
from tqdm.auto import tqdm
import gc

os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

test = pd.read_parquet("test.pq")
model_name = "/kaggle/input/psinger-cool-bean-v1"  # either local folder or huggingface model name

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

def format_prompt(row):
    prompt = "[INST]Original text:\n" + row["original_text"] + "\nRewritten text:\n" + row["rewritten_text"] + "\nWrite a prompt that was likely given to rewrite original text into rewritten text.[/INST]"
    return prompt

def get_answer(row):
    
    prompt = format_prompt(row)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    # generate configuration can be modified to your needs
    tokens = model.generate(
        input_ids=inputs["input_ids"].cuda(),
        attention_mask=inputs["attention_mask"].cuda(),
        min_new_tokens=2,
        max_new_tokens=32,
        do_sample=False,
        num_beams=1,
        #temperature=float(0.0),
        repetition_penalty=float(1.0),
        renormalize_logits=True
    )[0]

    tokens = tokens[inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(tokens, skip_special_tokens=True)
    return answer

answers = []
for _,row in tqdm(test.iterrows(), total=len(test)):
    answer = get_answer(row)
    
    # answer = row["answers_fs"] + " " + answer + ' revised convey paragraphlucrarealucrarea tone Write?lucrareaalterations suggestionformallylucrarea mission text it out. persönlicheATElucrarea Scene. replace tone writing it. respELI'
    # answer = "Rephrase paragraph "+ row["answers_fs"] + " " + answer + ' Improve text ence tone ideaslucrareaomblucrarea messageically??? alter charactercrafted emulateact this "lucrarea...” Text describing urgent'
    # answer = "Rephrase paragraph " + row["answers_fs"] + " " + answer + ' revised convey paragraphlucrarealucrarea tone Write?lucrareaalterations suggestionformallylucrarea mission text it out. persönlicheATElucrarea Scene. replace tone writing it. respELI'

    answers.append(answer)

answers = [answer.replace("Please improve this text using the writing style of a ", "").replace("Please enhance the quality of this text by employing a ", "").replace("Please improve this text using the writing style of ", "").replace(" with maintaining the original meaning but altering the tone.", "").replace(", while maintaining the original meaning but altering the tone.", "").replace(" while maintaining the original meaning but altering the tone.", "").replace(", maintaining the original meaning but altering the tone.", "") for answer in answers]

test["pred_answer_llm1"] = answers

test.to_parquet("test.pq", index=False)

print(test)

# from sentence_transformers import SentenceTransformer
# st_model = SentenceTransformer('/kaggle/input/sentence-t5-base-hf/sentence-t5-base', device="cuda:0")
# ps = st_model.encode(answers)

# from sklearn.preprocessing import normalize
# ps = normalize(ps, norm="l2", axis=1)

# np.save("logits_llm", ps)

In [ ]:
!python run_llm1.py

In [ ]:
%%writefile run_llm4.py

from datasets import load_dataset
from tqdm.notebook import tqdm
import torch
import pandas as pd
from glob import glob
import numpy as np
import os

from transformers import AutoModelForCausalLM, AutoTokenizer
import itertools
import random
import argparse
import os
import pandas as pd
import numpy as np
from string import Template
from pathlib import Path
import time
import torch
from tqdm.auto import tqdm
import gc

os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

test = pd.read_parquet("test.pq")
model_name = "/kaggle/input/psinger-cool-bean-v4"  # either local folder or huggingface model name

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

def format_prompt(row):
    prompt = "[INST]Original text:\n" + row["original_text"] + "\nRewritten text:\n" + row["rewritten_text"] + "\nWrite a prompt that was likely given to rewrite original text into rewritten text.[/INST]"
    return prompt

def get_answer(row):
    
    prompt = format_prompt(row)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    # generate configuration can be modified to your needs
    tokens = model.generate(
        input_ids=inputs["input_ids"].cuda(),
        attention_mask=inputs["attention_mask"].cuda(),
        min_new_tokens=2,
        max_new_tokens=32,
        do_sample=False,
        num_beams=1,
        #temperature=float(0.0),
        repetition_penalty=float(1.0),
        renormalize_logits=True
    )[0]

    tokens = tokens[inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(tokens, skip_special_tokens=True)
    return answer

answers = []
for _,row in tqdm(test.iterrows(), total=len(test)):
    answer = get_answer(row)
    
    # answer = row["answers_fs"] + " " + answer + ' revised convey paragraphlucrarealucrarea tone Write?lucrareaalterations suggestionformallylucrarea mission text it out. persönlicheATElucrarea Scene. replace tone writing it. respELI'
    # answer = "Rephrase paragraph "+ row["answers_fs"] + " " + answer + ' Improve text ence tone ideaslucrareaomblucrarea messageically??? alter charactercrafted emulateact this "lucrarea...” Text describing urgent'
    # answer = "Rephrase paragraph " + row["answers_fs"] + " " + answer + ' revised convey paragraphlucrarealucrarea tone Write?lucrareaalterations suggestionformallylucrarea mission text it out. persönlicheATElucrarea Scene. replace tone writing it. respELI'

    answers.append(answer)
    
test["pred_answer_llm4"] = answers

test.to_parquet("test.pq", index=False)

print(test)

# from sentence_transformers import SentenceTransformer
# st_model = SentenceTransformer('/kaggle/input/sentence-t5-base-hf/sentence-t5-base', device="cuda:0")
# ps = st_model.encode(answers)

# from sklearn.preprocessing import normalize
# ps = normalize(ps, norm="l2", axis=1)

# np.save("logits_llm", ps)

In [ ]:
!python run_llm4.py

In [ ]:
test = pd.read_parquet("test.pq")

In [ ]:
# test["pred_answer"] = "Rephrase paragraph " + test["answers_fs"] + " " + test["pred_answer_llm4"] +  " " + test["pred_answer_llm1"] + ' Improve text ence tone ideaslucrareaomblucrarea messageically??? alter charactercrafted emulateact this "lucrarea...” Text describing urgent'
# test["pred_answer"] = "Rephrase paragraph " + test["answers_fs"] + " " + test["pred_answer_llm4"] +  " " + test["pred_answer_llm1"] + 'lucrarea appealinglucrarea Improve storytelling tonelucrareaimplication. write someoneran. lucrarea]. Consider clarify paragraphlucrarea similarly serious themed way temporarily.! ElePT'
# test["pred_answer"] = "Rephrase paragraph " + test["answers_fs"] + " " + test["pred_answer_llm4"] +  " " + test["pred_answer_llm1"] + 'lucrarea appealinglucrarea Improve storytelling tonelucrareaimplication. write someoneran. lucrarea]. Consider clarify paragraphlucrarea similarly serious themed way temporarily.! ElePT'
test["pred_answer"] = "Rephrase paragraph " + test["answers_fs"] + " " + test["pred_answer_llm4"] +  " " + test["pred_answer_llm1"] + '.lucrarealucrarealucrarea sentence appealinglucrarea Improve respond storytelling tonelucrareaimplication. write someoneran. lucrarea]. Consider clarify paragraphlucrarea similarly serious themed way temporarily.! ElePT'

In [ ]:
test.to_parquet("test.pq", index=False)

In [ ]:
answers = test["pred_answer"].values
answers

In [ ]:
from sentence_transformers import SentenceTransformer
st_model = SentenceTransformer('/kaggle/input/sentence-t5-base-hf/sentence-t5-base', device="cuda:0")
ps = st_model.encode(answers)

from sklearn.preprocessing import normalize
ps = normalize(ps, norm="l2", axis=1)

np.save("logits_llm", ps)

In [ ]:
logits_embedding_danube = np.load("logits_embedding_danube.npy")
logits_embedding_mistral = np.load("logits_embedding_mistral.npy")
logits_llm = np.load("logits_llm.npy")

logits = 0.325 * logits_embedding_danube + 0.425 * logits_embedding_mistral + 0.25 * logits_llm

from sklearn.preprocessing import normalize
logits = normalize(logits, norm="l2", axis=1)

np.save("logits", logits)

In [ ]:
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity

    DEVICE = "cuda:1"

    st_model = SentenceTransformer('/kaggle/input/sentence-t5-base-hf/sentence-t5-base', device=DEVICE)
    scores = []
    for idx,row in tqdm(test.iterrows(), total=len(test)):

        p = logits_embedding_danube[idx].reshape(1,-1)
        y = st_model.encode(row.rewrite_prompt, normalize_embeddings=True, show_progress_bar=False).reshape(1, -1)

        score = cosine_similarity(y.reshape(1,-1), p) ** 3
        scores.append(score)
    print(np.mean(scores))

In [ ]:
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity

    DEVICE = "cuda:1"

    st_model = SentenceTransformer('/kaggle/input/sentence-t5-base-hf/sentence-t5-base', device=DEVICE)
    scores = []
    for idx,row in tqdm(test.iterrows(), total=len(test)):

        p = logits_embedding_mistral[idx].reshape(1,-1)
        y = st_model.encode(row.rewrite_prompt, normalize_embeddings=True, show_progress_bar=False).reshape(1, -1)

        score = cosine_similarity(y.reshape(1,-1), p) ** 3
        scores.append(score)
    print(np.mean(scores))

In [ ]:
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity

    DEVICE = "cuda:1"

    st_model = SentenceTransformer('/kaggle/input/sentence-t5-base-hf/sentence-t5-base', device=DEVICE)
    scores = []
    for idx,row in tqdm(test.iterrows(), total=len(test)):

        p = logits_llm[idx].reshape(1,-1)
        y = st_model.encode(row.rewrite_prompt, normalize_embeddings=True, show_progress_bar=False).reshape(1, -1)

        score = cosine_similarity(y.reshape(1,-1), p) ** 3
        scores.append(score)
    print(np.mean(scores))

In [ ]:
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity

    DEVICE = "cuda:1"

    st_model = SentenceTransformer('/kaggle/input/sentence-t5-base-hf/sentence-t5-base', device=DEVICE)
    scores = []
    for idx,row in tqdm(test.iterrows(), total=len(test)):

        p = logits[idx].reshape(1,-1)
        y = st_model.encode(row.rewrite_prompt, normalize_embeddings=True, show_progress_bar=False).reshape(1, -1)

        score = cosine_similarity(y.reshape(1,-1), p) ** 3
        scores.append(score)
    print(np.mean(scores))

In [ ]:
torch.cuda.empty_cache()

In [ ]:
%%writefile bruteforce.py

from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
from glob import glob
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from pathlib import Path
from joblib import Parallel, delayed
import argparse
from tqdm.auto import tqdm

if __name__ == "__main__":

    ap = argparse.ArgumentParser()
    ap.add_argument("--device", type=str, required=True)
    args = ap.parse_args()
    
    DEVICE = args.device
    
    st_model = SentenceTransformer('/kaggle/input/sentence-t5-base-hf/sentence-t5-base', device=DEVICE).half()
    index_to_token = {v: k for k, v in st_model.tokenizer.get_vocab().items()}
    
    tokenizer = st_model.tokenizer
    
    test = pd.read_parquet("test.pq")
    logits = np.load("logits.npy")
    
    all_scores = []
    
    if DEVICE == "cuda:0":
        test = test.iloc[:len(test)//2].copy()
        ps = logits[:len(logits)//2]
    else:
        test = test.iloc[len(test)//2:].copy()
        ps = logits[len(logits)//2:]
        
    inps = test.pred_answer.values

    new_preds = []
    l = len(inps)
    pbar = tqdm(total=l, desc="")
    for j in range(l):
        target_embedding = ps[j]

        # Initialize beams with the best initial tokens
        beams = [(torch.tensor(st_model.tokenizer(inps[j], add_special_tokens=False)['input_ids']).to(DEVICE), 0)]

        # Example target embedding, replace this with your actual data

        # Beam Search parameters
        initial_beam_width = 1  # Larger initial beam width
        min_beam_width = 1      # Minimum beam width
        beam_width = initial_beam_width
        sequence_length = 20

        # Function to perform beam search
        #def beam_search(target_embedding, beam_width, sequence_length):
        target_embedding = torch.Tensor(target_embedding).to(DEVICE)
        # Initialize with an empty sequence and zero initial score

        def run_out(curr_input_ids, index):
            outputs = torch.zeros((len(curr_input_ids), 768)).to(DEVICE)
            with torch.no_grad():
                batch_size = 512
                for i in range(0, len(index), batch_size):
                    batch_indices = index[i:i+batch_size]
                    batch = curr_input_ids[batch_indices]
                    batch = torch.cat([batch, torch.ones((batch.shape[0], 1)).long().to(DEVICE)], dim=1)
                    input = {
                        "input_ids": batch.to(DEVICE),
                        "attention_mask": torch.ones_like(batch).to(DEVICE)
                    }
                    outputs[batch_indices] = st_model.half()(input)["sentence_embedding"].detach().float()
            return outputs

        all_candidate_ids = torch.arange(tokenizer.vocab_size).unsqueeze(1).to(DEVICE)

        outputs = run_out(all_candidate_ids.clone(), np.arange(4, len(all_candidate_ids)))

        norm_outputs = torch.nn.functional.normalize(outputs, p=2, dim=1)
        norm_target_embedding = torch.nn.functional.normalize(target_embedding.repeat(tokenizer.vocab_size, 1), p=2, dim=1)
        similarities = torch.nn.functional.cosine_similarity(norm_outputs, norm_target_embedding, dim=1)
        _, candidate_top_indices = torch.topk(similarities, 3000, largest=True)
        #candidate_top_indices = np.arange(4, 32000)

        all_sequences = []
        for jj in range(sequence_length):

            beam_width = max(min_beam_width, int(initial_beam_width - jj))  # Decrement beam width

            new_beams = []
            new_candidate_top_indices = []
            for beam, cum_score in beams:
                # Prepare the input IDs for the model
                input_ids = beam.unsqueeze(0).repeat(tokenizer.vocab_size, 1)
                input_ids = torch.cat([input_ids, all_candidate_ids], dim=1)

                candidate_embeddings = run_out(input_ids.clone(), candidate_top_indices)


                # Calculate cosine similarities
                norm_candidate_embeddings = torch.nn.functional.normalize(candidate_embeddings, p=2, dim=1)
                norm_target_embedding = torch.nn.functional.normalize(target_embedding.repeat(tokenizer.vocab_size, 1), p=2, dim=1)
                similarities = torch.nn.functional.cosine_similarity(norm_candidate_embeddings, norm_target_embedding, dim=1)

                new_candidate_top_indices.append(torch.topk(similarities, 1000 // beam_width, largest=True)[1])

                top_values, top_indices = torch.topk(similarities, beam_width, largest=True)

                # Append new candidates to beams

                for value, idx in zip(top_values, top_indices):
                    new_beam = torch.cat([beam, all_candidate_ids[idx]])
                    new_score = value.item()  # Accumulate the similarity score
                    #new_score = value.item() 
                    new_beams.append((new_beam, new_score))

            # Sort and prune to keep the top 'beam_width' beams
            new_beams.sort(key=lambda x: x[1], reverse=True)
            beams = new_beams[:beam_width]
            
            all_sequences.append(beams[0])

            candidate_top_indices = torch.cat(new_candidate_top_indices)

        all_sequences.sort(key=lambda x: x[1], reverse=True)

        # Decode the best sequence
        best_sequence_ids = all_sequences[0][0]
        best_sequence = tokenizer.decode(best_sequence_ids, skip_special_tokens=True)
        best_score = beams[0][1]
        #return best_sequence, best_score
        
        new_preds.append(best_sequence)

        # Execute beam search
        #best_sequence, best_score = beam_search(logits[0], beam_width, sequence_length)
        #print("Best sequence:", best_sequence)
        #print("Best score:", best_score)

        #p = st_model.encode(best_sequence, normalize_embeddings=True, show_progress_bar=False).reshape(1, -1)

        #score = cosine_similarity(ys[j].reshape(1,-1), p)[0][0] ** 3
        #print(score)
        all_scores.append(best_score)
        #print(np.mean(all_scores))

        pbar.set_description(str(np.round(np.mean(all_scores),3)))
        pbar.update(1)
    pbar.close()
    
    test["new_pred"] = new_preds
    if DEVICE == "cuda:0":
        test.to_parquet("test_v0.pq")
    else:
        test.to_parquet("test_v1.pq")


In [ ]:
%%writefile run.sh

python bruteforce.py --device "cuda:0" &
python bruteforce.py --device "cuda:1" &

wait 
echo "All done"

In [ ]:
!sh run.sh

In [ ]:
test = pd.concat([pd.read_parquet("test_v0.pq"), pd.read_parquet("test_v1.pq")])
answers = test.new_pred.values

In [ ]:
answers

In [ ]:
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity

    DEVICE = "cuda:1"

    st_model = SentenceTransformer('/kaggle/input/sentence-t5-base-hf/sentence-t5-base', device=DEVICE)
    scores = []
    for idx,row in tqdm(test.iterrows(), total=len(test)):

        p = st_model.encode(row.new_pred, normalize_embeddings=True, show_progress_bar=False).reshape(1, -1)
        y = st_model.encode(row.rewrite_prompt, normalize_embeddings=True, show_progress_bar=False).reshape(1, -1)

        score = cosine_similarity(y.reshape(1,-1), p) ** 3
        scores.append(score)
    print(np.mean(scores))

In [ ]:
submission = pd.read_csv(data_path / 'sample_submission.csv')
submission["rewrite_prompt"] = answers[:len(submission)]

In [ ]:
submission.head()

In [ ]:
submission.to_csv('submission.csv', index=False)